In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# gender showed almost no effect in the EDA, so it is excluded from the features
X = df.drop(columns=["Churn", "gender"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

(5634, 18) (1409, 18)


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X_train.columns if c not in num_cols + ["SeniorCitizen"]]

# KNN and Logistic Regression: scale the numbers and one-hot encode the categories
prep_scaled = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

# Random Forest: one-hot encode only (trees don't need scaling)
prep_tree = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    "KNN": Pipeline([("prep", prep_scaled), ("clf", KNeighborsClassifier())]),
    "Logistic Regression": Pipeline([("prep", prep_scaled), ("clf", LogisticRegression(max_iter=1000))]),
    "Random Forest": Pipeline([("prep", prep_tree), ("clf", RandomForestClassifier(random_state=42))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

baseline = pd.DataFrame(rows).T.round(3)
baseline

,accuracy,precision,recall,f1,roc_auc
KNN,0.770,0.572,0.528,0.549,0.780
Logistic Regression,0.802,0.654,0.544,0.593,0.846
Random Forest,0.788,0.632,0.481,0.546,0.819


**Baseline results (5-fold stratified CV, default hyperparameters)**

- Logistic Regression achieved the best baseline performance across all metrics (F1 0.593, ROC-AUC 0.846), suggesting the churn signal is largely linear after scaling and one-hot encoding.
- Random Forest had the lowest recall (0.481), and KNN the lowest ROC-AUC (0.780).
- Recall is only about 0.48-0.54 for all models: each misses roughly half of the churners. Because churners are only 26.5% of the data, the models favour the majority class, so accuracy (0.77-0.80) overstates real performance. Class-imbalance handling and tuning against F1/recall will be explored next.